In [1]:
import pandas as pd
output_csv = "/home/server/Projects/data/AKI/preop_data_test.csv"
df = pd.read_csv(output_csv)

In [1]:
import pandas as pd
import numpy as np


file = '/home/server/Projects/data/AKI/tabular_combined_unnormalized.csv'
df = pd.read_csv(file)

df_isna = df[df.filter(like='isna').columns]
col_fill_rate = 1 - (df_isna.sum(axis=0) / df_isna.shape[0])
col_fill_rate.index = col_fill_rate.index.str[:-5]

# cut down on all the repeat columns from taking eight metrics for each of these regular variables
high_frequency_labels = ["rr", "hr", "spo2", "fio2", "pmean", "etco2", "peep", 
"pip", "art_mbp", "cpat", "vt", "art_sbp", "art_dbp", 
"minvol", "pplat", "bt", "etgas", "cvp"]
medium_frequency_labels = ["pap_mbp", "pap_sbp", "pap_dbp", "nibp_mbp", "nibp_dbp", "nibp_sbp"]
regular_labels = high_frequency_labels + medium_frequency_labels
for label in regular_labels:
    cols = col_fill_rate.filter(like=label)
    value = min(cols)
    for col in cols.keys():
        col_fill_rate.pop(col)
    col_fill_rate[label] = value

# cleaning labels
col_fill_rate.index = col_fill_rate.index.str.replace(r"(mean_|sum_)", "", regex=True)

In [2]:
for i, fr in col_fill_rate.items():
    print(i, '\t', fr)

last_preop_scr 	 0.9999826761832167
min_preop_scr 	 0.9999826761832167
preop_total_protein 	 0.9640011087242741
preop_sodium 	 0.9823470306978034
preop_potassium 	 0.9824163259649366
preop_platelet 	 0.9742568082599958
preop_glucose 	 0.942900699882198
preop_wbc 	 0.975867923220844
preop_alt 	 0.9646767375788233
preop_chloride 	 0.9732866745201303
preop_lymphocyte 	 0.9266682835562331
preop_phosphorus 	 0.9527233039983369
preop_albumin 	 0.9674485482641536
preop_fibrinogen 	 0.8527995287921835
preop_creatinine 	 1.0
preop_ptinr 	 0.9606922597186612
preop_total_bilirubin 	 0.9338749913380916
preop_alp 	 0.963273508419375
preop_aptt 	 0.9570542581941653
preop_calcium 	 0.9684013581872358
preop_bun 	 0.9701510636823505
preop_ast 	 0.9649539186473564
preop_crp 	 0.7919929318827524
preop_hb 	 0.9770459427621093
preop_hct 	 0.9818792876446538
preop_seg 	 0.9265296930219666
air 	 0.7957002286743815
bis 	 0.6260654147321738
cbro2 	 0.0031182870209964797
ci 	 0.11885870695031531
dobui 	 0.02664

In [1]:
import pandas as pd
import numpy as np


file = '/home/server/Projects/data/AKI/tabular_combined_unnormalized.csv'
df = pd.read_csv(file)

# one row per subject, not operation
df = df.groupby('subject_id').agg('first').reset_index()
df = df.drop(columns=['op_id'])

S = ''
def sprint(s):
    global S
    S = S + s + '\n'

In [2]:
sprint(f'Total Population:\t{df.shape[0]}')
sprint('\nmetric\tmean, std')
#misc
mn_sd_cols = ['age', 'weight', 'height', 'BMI', 
              'BSA', 'asa', 'num_card_events', 
              'booking_case_length']
for col in mn_sd_cols:
    mn = df[col].mean()
    sd = df[col].std()
    sprint(f'{col}:\t{mn:.2f} ± {sd:.2f}')

sprint('\nmetric\tN\tpercent')

# sex
males = df.sex.value_counts()[True]
females = df.sex.value_counts()[False]
sprint(f'Male:\t{males}\t{males / df.shape[0] * 100:.2f}%')
sprint(f'Female:\t{females}\t{females / df.shape[0] * 100:.2f}%')

# asa
desc = df.asa.value_counts()
for key in desc.keys():
    value = desc[key]
    sprint(f'ASA {key}:\t{value}\t{value / df.shape[0] * 100:.3f}%')

# aki
desc = df.aki_boolean.value_counts()
value = desc[True]
sprint(f'AKI positivity:\t{value}\t{value / df.shape[0] * 100:.3f}%')

# departments
dept_cols = [col for col in df.columns if 'department' in col]
for col in dept_cols:
    value = df[col].sum()
    sprint(f'{col}:\t{value}\t{value / df.shape[0] * 100:.4f}%')

In [3]:
print(S)

Total Population:	47211

metric	mean, std
age:	58.18 ± 15.10
weight:	63.03 ± 11.71
height:	162.36 ± 9.00
BMI:	23.87 ± 3.63
BSA:	1.68 ± 0.19
asa:	1.83 ± 0.64
num_card_events:	0.89 ± 3.96
booking_case_length:	230.67 ± 121.93

metric	N	percent
Male:	24978	52.91%
Female:	22233	47.09%
ASA 2.0:	27685	58.641%
ASA 1.0:	13876	29.391%
ASA 3.0:	5266	11.154%
ASA 4.0:	357	0.756%
ASA 5.0:	27	0.057%
AKI positivity:	2891	6.124%
department_AN:	2	0.0042%
department_CTS:	6786	14.3738%
department_DM:	1	0.0021%
department_EM:	2	0.0042%
department_GS:	13866	29.3703%
department_IM:	16	0.0339%
department_NS:	7596	16.0895%
department_OG:	964	2.0419%
department_OL:	1098	2.3257%
department_OS:	10802	22.8803%
department_OT:	147	0.3114%
department_PED:	4	0.0085%
department_PS:	815	1.7263%
department_RAD:	283	0.5994%
department_RO:	0	0.0000%
department_UR:	4829	10.2285%



In [7]:
df[df['department_PED'] == 1]

,subject_id,age,sex,height,weight,asa,emop,BSA,BMI,booking_case_length,...,sum_ftn_isna,sum_n2o_isna,sum_pc_isna,sum_pheresis_isna,sum_rbc_isna,sum_uo_isna,fluids_agg_isna,equiv_MAC_totals_isna,aki,aki_boolean
10573,122221734.0,25.0,True,180.0,65.0,2.0,0.0,1.802776,20.061728,130.0,...,True,True,True,True,True,True,False,True,0.00,False
18113,138174432.0,40.0,False,155.0,50.0,3.0,0.0,1.467235,20.811655,410.0,...,True,True,True,True,True,False,False,False,-0.08,False
32619,169030610.0,35.0,True,175.0,80.0,2.0,0.0,1.972027,26.122449,510.0,...,False,True,True,True,True,False,False,False,0.00,False
36269,176846594.0,35.0,True,180.0,85.0,3.0,0.0,2.061553,26.234568,160.0,...,True,True,True,True,True,True,False,True,0.03,False
